> **Registro histórico, no se corre.** Así se entrenó exp002 el 23 sep con `src/knee/train.py`, que ya no existe.
>
> La misma receta, ahora con el entrenamiento en celdas, está en `experiments/exp003_ref_split_v1/`.

# 03 · Baseline DINOv2-small

Entrena un clasificador por estudio desde el cache, usando pseudo-etiquetas si existen. Un fold suele tardar varias horas; usa una GPU de 48 GB o más. Guarda checkpoint, OOF, métricas y curvas en tu carpeta de runs.

In [ ]:
# CONFIG
RUN_NAME = "dinov2s_baseline_v1"
CACHE_ROOT = "/workspace/data/cache/v1_256"
RAW_ROOT = "/workspace/data/raw"
LABELS_CSV = "/workspace/data/labels/report_labels_v1.csv"
BACKBONE_PATH = "/workspace/models/dinov2-small"
FOLD = 0
SELECT_ON = "pseudo"
FOLDS_TO_RUN = []
IMAGE_SIZE = 224
MAX_SERIES = 5
N_SLICES = 16
BATCH_SIZE = 2
EPOCHS = 5
LR = 1e-4
BACKBONE_LR = 1e-5
GRAD_ACCUM = 4
N_WORKERS = 4
FREEZE_EPOCHS = 1
NOTES = "baseline"


In [ ]:
from pathlib import Path
import os
import sys
import warnings
import matplotlib.pyplot as plt
import pandas as pd

repo = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo))
from src.knee.config import LABELS, user_dir
from src.knee.train import train_fold

def read_train_csv(path):
    for encoding in ("utf-8", "cp1252", "latin-1"):
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            pass
    raise RuntimeError(f"No se pudo leer {path}")

gold = read_train_csv(Path(RAW_ROOT) / "train.csv")
gold_rows = gold.dropna(subset=LABELS, how="all")[["StudyInstanceUID", *LABELS]]
run_dir = user_dir() / "runs" / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
if Path(LABELS_CSV).exists():
    labels = pd.read_csv(LABELS_CSV)[["StudyInstanceUID", *LABELS]]
    gold_path = run_dir / "gold_labels.csv"
    gold_rows.to_csv(gold_path, index=False)
    GOLD_CSV = str(gold_path)
    print(f"Entrenando con {len(labels)} estudios pseudo y evaluando {len(gold_rows)} estudios gold por separado.")
else:
    warnings.warn("No existe el archivo de pseudo-etiquetas. Se usarán solo los 58 estudios gold.")
    fallback = run_dir / "gold_only_labels.csv"
    gold_rows.to_csv(fallback, index=False)
    LABELS_CSV = str(fallback)
    GOLD_CSV = None


In [ ]:
folds = FOLDS_TO_RUN or [FOLD]
all_metrics = {}
for fold in folds:
    fold_dir = user_dir() / "runs" / RUN_NAME / f"fold_{fold}"
    cfg = {
        "run_name": RUN_NAME, "run_dir": str(fold_dir), "fold": fold,
        "labels_csv": LABELS_CSV, "gold_csv": GOLD_CSV,
        "cache_index": str(Path(CACHE_ROOT) / "index.csv"),
        "backbone_path": BACKBONE_PATH, "image_size": IMAGE_SIZE,
        "max_series": MAX_SERIES, "n_slices": N_SLICES, "batch_size": BATCH_SIZE,
        "epochs": EPOCHS, "lr": LR, "backbone_lr": BACKBONE_LR, "grad_accum": GRAD_ACCUM,
        "workers": N_WORKERS, "freeze_epochs": FREEZE_EPOCHS,
        "gradient_checkpointing": True, "select_on": SELECT_ON, "notes": NOTES,
    }
    all_metrics[fold] = train_fold(cfg)


In [ ]:
for fold, result in all_metrics.items():
    history = pd.DataFrame(result["history"])
    for source in ("pseudo", "gold"):
        metrics = result[f"{source}_metrics"]
        if metrics is not None:
            table = pd.DataFrame(metrics["per_label"].items(), columns=["label", "auc"])
            table.insert(0, "source", source)
            display(table)
    history.plot(x="epoch", y=["pseudo_auc", "gold_auc"], marker="o", title=f"Fold {fold}")
    plt.show()
